# BIST Hacim Tarayıcısı

**TradingView Scanner API** ile BIST hisselerinde **alım ilgisi** ve **satış baskısı** taraması.

> Not: Bu net alış/satış verisi değildir. Fiyat yönü × hacim ile piyasa ilgisi/baskısı tahminidir.

## Endpoint

```
POST https://scanner.tradingview.com/turkey/scan?label-product=screener-stock
```

**Zorunlu header'lar:** `Origin` ve `Referer` (yoksa 403 dönebilir).

## İstek (Request) Alanları

| Alan | Açıklama |
|------|----------|
| `columns` | Dönecek alan listesi |
| `filter` | Ana filtre · senaryoyu belirler (change vs 0) |
| `sort` | Sıralama · `{sortBy, sortOrder}` |
| `range` | Sonuç dilimi · `[start, end]` |
| `markets` | Piyasa filtresi · `['turkey']` |
| `filter2` | Tip zarfı · stock + common, pre-ipo hariç |
| `options.lang` | Yanıt dili · `'en'` veya `'tr'` |

## Operatörler

- `egreater` → büyük veya eşit (≥)
- `less` → küçük (<)
- `equal` → eşit (=)
- `has` → listede içerir
- `has_none_of` → listede hiçbirini içermez

## Senaryolar

- **Alım ilgisi:** `change ≥ 0`, hacme göre azalan ilk 3
- **Satış baskısı:** `change < 0`, hacme göre azalan ilk 3

## Yanıt (Response) Yapısı

`{ totalCount, data[] }` → `data`, kriterlere uyan BIST hisselerinin sıralı listesi.

Her `data[i]` öğesi:
- `s` → Sembol (örn. `'BIST:THYAO'`)
- `d[]` → `columns` sırasına göre değerler dizisi
  - `d[0]` ticker-view (sembol meta)
  - `d[1]` close (son fiyat, TL)
  - `d[2]` change (yüzde değişim)
  - `d[3]` volume (günlük işlem hacmi)
  - `d[4]` market_cap_basic (piyasa değeri, TL)

## 1. Ortak Ayarlar

In [1]:
import requests

URL = "https://scanner.tradingview.com/turkey/scan?label-product=screener-stock"

HEADERS = {
    "content-type": "application/json",
    "origin": "https://www.tradingview.com",
    "referer": "https://www.tradingview.com/",
}

COLUMNS = ["ticker-view", "close", "change", "volume", "market_cap_basic"]

FILTER2 = {
    "operator": "and",
    "operands": [
        {"operation": {"operator": "and", "operands": [
            {"expression": {"left": "type", "operation": "equal", "right": "stock"}},
            {"expression": {"left": "typespecs", "operation": "has", "right": ["common"]}},
        ]}},
        {"expression": {"left": "typespecs", "operation": "has_none_of", "right": ["pre-ipo"]}},
    ],
}

## 2. Tarama Fonksiyonu

`operation` parametresi senaryoyu belirler:
- `"egreater"` → alım ilgisi (change ≥ 0)
- `"less"` → satış baskısı (change < 0)

In [2]:
def tara(operation="egreater", limit=3):
    body = {
        "columns": COLUMNS,
        "filter": [{"left": "change", "operation": operation, "right": 0}],
        "sort": {"sortBy": "volume", "sortOrder": "desc"},
        "range": [0, limit],
        "markets": ["turkey"],
        "options": {"lang": "en"},
        "filter2": FILTER2,
    }
    r = requests.post(URL, headers=HEADERS, json=body)
    r.raise_for_status()
    return r.json()

## 3. Alım İlgisi · Hacme göre ilk 3 yükselen

In [3]:
alim = tara(operation="egreater", limit=3)

print(f"Toplam: {alim['totalCount']}\n")
print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}")
print("-" * 56)
for row in alim["data"]:
    sembol = row["s"]
    _, close, change, volume, _ = row["d"]
    print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}")

Toplam: 311

Sembol               Fiyat    Değişim              Hacim
--------------------------------------------------------
BIST:SASA             3.16      6.40%      3,844,541,500
BIST:CANTE            1.72      0.58%        398,694,270
BIST:EUREN            5.80      4.88%        397,432,810


## 4. Satış Baskısı · Hacme göre ilk 3 düşen

In [4]:
satis = tara(operation="less", limit=3)

print(f"Toplam: {satis['totalCount']}\n")
print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}")
print("-" * 56)
for row in satis["data"]:
    sembol = row["s"]
    _, close, change, volume, _ = row["d"]
    print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}")

Toplam: 259

Sembol               Fiyat    Değişim              Hacim
--------------------------------------------------------
BIST:ESEN             3.86     -9.60%        597,932,020
BIST:ISCTR           14.45     -2.30%        495,303,850
BIST:KONTR           10.53    -10.00%        194,619,212


## Notlar

- **Asimetri:** alım için `egreater` (≥ 0 dahil), satış için `less` (strict < 0).
- `range = [0, 3]` ilk 3, `[0, 50]` ilk 50 sonuç.
- `filter2` zarfı stock-only ekran için sabit: `type=stock`, `typespecs has 'common'`, `pre-ipo` hariç.
- `Origin` & `Referer` header'ları zorunlu — yoksa 403 dönebilir.